# Proof of Concept Experiment

In [ ]:
import pandas as pd
import plotnine as gg
from sklearn.manifold import smacof

from dataset_similarity.constants import METRICS_RESULT_DIR, PROJECT_DIR
from dataset_similarity.utils import load_yaml_from_path

Constants + fns

In [ ]:
# List of datasets in the results
NAME_TO_ALPHA_MAP = {
    "domainnet_clipart_1000": 0.0,
    "experiment_0_alpha_0.25": 0.25,
    "experiment_0_alpha_0.5": 0.5,
    "experiment_0_alpha_0.75": 0.75,
    "domainnet_real_1000": 1.0,
}
DATASETS = list(NAME_TO_ALPHA_MAP.keys())

# Metric labels
METRIC_LABEL_MAP = {
    "mmd": "MMD",
    "ot_exact": "OT (Exact)",
    "ot_sinkhorn": "OT (Sinkhorn)",
    "otdd_approx": "OTDD (Approx)",
    "otdd_exact": "OTDD (Exact)",
    "otce_ot_sinkhorn_both": "OTCE OT (Sinkhorn)",
    "otce_ot_sinkhorn_coupling": "F-OTCE OT (Sinkhorn)",
    "otce_otdd_both": "OTCE OTDD",
    "otce_otdd_coupling": "F-OTCE OTDD",
}
METRIC_LABEL_ORDER = METRIC_LABEL_MAP.keys()

In [ ]:
# Fn for populating a distance matrix for a given metric
def populate_distance_matrix(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    distance_matrix = pd.DataFrame(0.0, index=DATASETS, columns=DATASETS, dtype=float)
    for _, row in df.iterrows():
        d1, d2 = row["dataset1"], row["dataset2"]
        distance_matrix.loc[d1, d2] = row[metric]
        distance_matrix.loc[d2, d1] = row[metric]
    return distance_matrix

# Fn for producing MDS solutions for a given distance matrix
def compute_mds_solutions(df: pd.DataFrame, metric: str, max_dims=5):
    distance_matrix = populate_distance_matrix(df, metric)
    return [
        smacof(
            dissimilarities=distance_matrix.to_numpy(),
            metric=True,
            n_components=i,
        )
        for i in range(1, max_dims + 1)
    ]

Load data

In [ ]:
# Directory for plot results
plots_dir = PROJECT_DIR / "plots"
plots_dir.mkdir(exist_ok=True)

In [ ]:
# Load results data
experiment_results_dir = METRICS_RESULT_DIR / "experiment_0_poc"

# Read all metric results into a list of dicts and create a DataFrame
results = pd.DataFrame(
    [
        load_yaml_from_path(yaml_path)
        for yaml_path in experiment_results_dir.glob("*.yaml")
    ]
)

Generate smacof results for all metrics

In [ ]:
mds_solutions = {
    metric: compute_mds_solutions(results, metric)
    for metric in results.columns if metric not in ["dataset1", "dataset2"]
}

In [ ]:
_, x = mds_solutions["mmd"][0]
x

## Plot Stress

In [ ]:
stress_df = pd.DataFrame(
    {"ndim": i+1, "metric": metric, "stress": x[1]}
    for metric, result in mds_solutions.items()
    for i, x in enumerate(result)
)
stress_df["metric_label"] = pd.Categorical(
    stress_df["metric"].map(METRIC_LABEL_MAP),
    categories=[METRIC_LABEL_MAP[m] for m in METRIC_LABEL_ORDER],
)

(
    gg.ggplot(stress_df, gg.aes(x="ndim", y="stress"))
    + gg.geom_line()
    + gg.geom_point()
    + gg.facet_wrap("~metric_label", scales="free_y")
    + gg.labs(
        title="MDS Stress vs. Number of Dimensions",
        x="Number of Dimensions",
        y="Stress",
    )
    + gg.theme_bw()
)

## 2D Solutions

In [ ]:
smacof_coords = pd.DataFrame(
    {"metric": metric, "dataset": DATASETS[i], "D1": coords[0], "D2": coords[1]}
    for metric, result in mds_solutions.items()
    for i, coords in enumerate(result[1][0]) # 2nd ele of results for 2D solutions
)
smacof_coords["metric_label"] = pd.Categorical(
    smacof_coords["metric"].map(METRIC_LABEL_MAP),
    categories=[METRIC_LABEL_MAP[m] for m in METRIC_LABEL_ORDER],
)
smacof_coords["alpha"] = pd.Categorical(
    smacof_coords["dataset"].map(NAME_TO_ALPHA_MAP),
    categories=sorted(NAME_TO_ALPHA_MAP.values())
)
pct = 0.1  # nudge up by 5% of each facet's y-range
smacof_coords["y_label"] = smacof_coords.groupby("metric_label")["D2"].transform(
    lambda x: x + pct * (x.max() - x.min())
)

(
    gg.ggplot(smacof_coords, gg.aes(x="D1", y="D2", label="alpha"))
    + gg.geom_point()
    + gg.geom_text(gg.aes(y="y_label"))
    + gg.facet_wrap("~metric_label", scales="free")
    + gg.coord_cartesian()
    + gg.labs(
        title="MDS Dimension 1 vs Dimension 2",
        x="Dimension 1 Value",
        y="Dimension 2 Value",
    )
    + gg.theme_bw()
)